# NB4 — FastAPI Server + MQTT Pipeline

The full cloud backend in one file.

**7 REST endpoints:**

| Endpoint | Powers |
|---|---|
| `GET /devices/{id}/health` | Homeowner gauge |
| `GET /devices/{id}/telemetry` | Trend charts |
| `GET /devices/{id}/alerts` | Alert history |
| `GET /reports/{id}` | Engineer diagnostic |
| `POST /devices/{id}/report` | On-demand RAG |
| `POST /simulate/whatif` | Physics simulator |
| `GET /fleet/overview` | Luminous dashboard |

**Swap markers:** `[SWAP_MQTT]` `[SWAP_DB]` `[SWAP_PUSH]`

**Run server:** `python notebook4_api_server.py --server` then open `http://localhost:8000/docs`

**ARIA — Relay Intelligence · APOGEE'26 · BITS Pilani · Luminous Power Technologies**

## Installation

In [ ]:
!pip install fastapi uvicorn pydantic scikit-learn joblib numpy pandas
# Notebook 3 must be in the same directory (imports from it)

## Setup

In [ ]:
"""
ARIA — Adaptive Relay Intelligence & Anomaly System
Notebook 4: FastAPI Server + MQTT Pipeline

This is the cloud backend. Four responsibilities:
  1. MQTT subscriber  — receives real-time inverter data from dongles
  2. Inference engine — runs GBR + Isolation Forest on incoming data
  3. RAG trigger      — calls Notebook 3 pipeline when alert fires
  4. REST API         — serves the mobile app

Run demo:   python notebook4_api_server.py
Run server: python notebook4_api_server.py --server
API docs:   http://localhost:8000/docs

Swap markers:
  [SWAP_MQTT]   paho-mqtt real broker
  [SWAP_DB]     TimescaleDB / PostgreSQL
  [SWAP_PUSH]   FCM push notifications
"""

import os, json, time, threading, joblib, warnings, sys
from datetime import datetime
from typing import Optional
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

sys.path.insert(0, os.path.dirname(__file__))
from notebook3_rag_copilot import (
    TFIDFRetriever, KB_DOCUMENTS, build_knowledge_base,
    generate_diagnostic_report
)

MODEL_DIR = "aria_models"
KB_DIR    = "aria_kb"
DATA_DIR  = "aria_data"

## Section

In [ ]:
# 1. MODEL LOADER

## Section

In [ ]:
class ModelStore:
    def __init__(self):
        self.gbr = self.iso = self.if_scaler = None
        self.if_feats = []
        self.conf_q = 9.0
        self.feature_cols = []
        reg_path = os.path.join(MODEL_DIR, "model_registry.json")
        if not os.path.exists(reg_path):
            print("  [WARN] Run Notebook 2 first to generate models")
            return
        with open(reg_path) as f:
            reg = json.load(f)
        self.feature_cols = reg.get("feature_cols", [])

        gbr_p = os.path.join(MODEL_DIR, reg.get("gbr", "gbr_rul.pkl"))
        if os.path.exists(gbr_p):
            self.gbr = joblib.load(gbr_p)
            print(f"  Loaded GBR ({len(self.feature_cols)} features)")

        if_p  = os.path.join(MODEL_DIR, reg.get("isolation_forest", "isolation_forest.pkl"))
        sc_p  = os.path.join(MODEL_DIR, reg.get("if_scaler", "if_scaler.pkl"))
        ft_p  = os.path.join(MODEL_DIR, reg.get("if_features", "if_features.json"))
        if os.path.exists(if_p):
            self.iso       = joblib.load(if_p)
            self.if_scaler = joblib.load(sc_p)
            with open(ft_p) as f:
                self.if_feats = json.load(f)
            print(f"  Loaded IsolationForest")

        conf_p = os.path.join(MODEL_DIR, "conformal_meta.json")
        if os.path.exists(conf_p):
            with open(conf_p) as f:
                self.conf_q = json.load(f)["q_hat"]

## Section

In [ ]:
# 2. IN-MEMORY STORE  [SWAP_DB]

## Section

In [ ]:
class InMemoryStore:
    """[SWAP_DB] Replace with TimescaleDB for production."""
    def __init__(self):
        self._lock      = threading.Lock()
        self._telemetry = {}   # device_id → list[dict]
        self._alerts    = {}   # device_id → list[dict]
        self._reports   = {}   # report_id → dict

    def push_telemetry(self, device_id, snap):
        with self._lock:
            bucket = self._telemetry.setdefault(device_id, [])
            bucket.append(snap)
            self._telemetry[device_id] = bucket[-200:]

    def get_telemetry(self, device_id, n=48):
        with self._lock:
            return list(self._telemetry.get(device_id, []))[-n:]

    def push_alert(self, device_id, alert):
        with self._lock:
            self._alerts.setdefault(device_id, []).append(alert)

    def get_alerts(self, device_id):
        with self._lock:
            return list(self._alerts.get(device_id, []))

    def save_report(self, report):
        with self._lock:
            self._reports[report["report_id"]] = report

    def get_report(self, rid):
        with self._lock:
            return self._reports.get(rid)

    def all_devices(self):
        with self._lock:
            return list(self._telemetry.keys())

## Section

In [ ]:
# 3. INFERENCE ENGINE

## Section

In [ ]:
def build_feature_row(history: list, feature_cols: list) -> pd.DataFrame:
    """Rebuild rolling features from device history — mirrors Notebook 2."""
    if not history:
        return pd.DataFrame()
    df = pd.DataFrame(history)
    df["timestamp"] = pd.to_datetime(df.get("timestamp", datetime.now().isoformat()))

    hi_col = "reg_4002_relay_health_index"
    if hi_col in df.columns:
        for w, lbl in [(12,"6h"),(48,"24h"),(min(len(df),144),"72h")]:
            w = min(w, len(df))
            df[f"hi_mean_{lbl}"]     = df[hi_col].rolling(w, min_periods=1).mean()
            df[f"hi_std_{lbl}"]      = df[hi_col].rolling(w, min_periods=1).std().fillna(0)
            df[f"bounce_mean_{lbl}"] = df.get("reg_4003_contact_bounce_ms",
                                              pd.Series([2.0]*len(df))).rolling(w, min_periods=1).mean()
            df[f"arc_sum_{lbl}"]     = df.get("reg_4005_arc_energy_this_switch",
                                              pd.Series([0.0]*len(df))).rolling(w, min_periods=1).sum()
            df[f"temp_max_{lbl}"]    = df.get("reg_3019_temp_C",
                                              pd.Series([30.0]*len(df))).rolling(w, min_periods=1).max()
    df["hi_delta"]       = df.get(hi_col, pd.Series([100.0]*len(df))).diff().fillna(0)
    df["hi_accel"]       = df["hi_delta"].diff().fillna(0)
    df["cumulative_arc"] = df.get("reg_4005_arc_energy_this_switch",
                                   pd.Series([0.0]*len(df))).cumsum()
    df["hour_sin"] = np.sin(2 * np.pi * df["timestamp"].dt.hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["timestamp"].dt.hour / 24)
    df["dow_sin"]  = np.sin(2 * np.pi * df["timestamp"].dt.dayofweek / 7)

    last = df.iloc[[-1]].copy()
    for col in feature_cols:
        if col not in last.columns:
            last[col] = 0.0
    return last[feature_cols].fillna(0)


def run_inference(snapshot: dict, history: list, models: ModelStore) -> dict:
    """Full inference for one incoming MQTT snapshot."""
    feat_row = build_feature_row(history + [snapshot], models.feature_cols)

    rul_pred = 180.0
    if models.gbr is not None and not feat_row.empty:
        try:
            rul_pred = float(np.clip(models.gbr.predict(feat_row)[0], 0, 365))
        except Exception as e:
            print(f"  [WARN] GBR: {e}")

    hi = float(snapshot.get("reg_4002_relay_health_index", 100.0))

    is_anomaly = False
    if models.iso is not None and models.if_feats:
        try:
            row    = np.array([[snapshot.get(f, 0.0) for f in models.if_feats]])
            scaled = models.if_scaler.transform(row)
            is_anomaly = models.iso.predict(scaled)[0] == -1
        except Exception as e:
            print(f"  [WARN] IF: {e}")

    alert_level = ("RED"    if hi < 10 or rul_pred < 7  else
                   "ORANGE" if hi < 30 or rul_pred < 30 else
                   "YELLOW" if hi < 60 or rul_pred < 90 else "GREEN")

    return {
        "rul_days":    round(rul_pred, 1),
        "rul_ci_low":  round(max(0, rul_pred - models.conf_q), 1),
        "rul_ci_high": round(rul_pred + models.conf_q, 1),
        "health_index":round(hi, 1),
        "is_anomaly":  is_anomaly,
        "alert_level": alert_level,
    }

## Section

In [ ]:
# 4. MQTT HANDLER  [SWAP_MQTT]

## Section

In [ ]:
class MQTTHandler:
    """
    [SWAP_MQTT] Replace with real paho-mqtt:
        import paho.mqtt.client as mqtt
        client = mqtt.Client()
        client.on_message = lambda c,u,m: self._handle(
            m.topic.split("/")[2], json.loads(m.payload))
        client.connect(BROKER_HOST, 1883)
        client.subscribe("aria/inverters/+/telemetry")
        client.loop_start()
    """
    def __init__(self, store: InMemoryStore, models: ModelStore,
                 retriever: TFIDFRetriever):
        self.store     = store
        self.models    = models
        self.retriever = retriever

    def handle(self, device_id: str, payload: dict):
        payload.setdefault("timestamp", datetime.now().isoformat())
        history = self.store.get_telemetry(device_id, n=150)
        result  = run_inference(payload, history, self.models)
        enriched = {**payload, **result, "device_id": device_id}
        self.store.push_telemetry(device_id, enriched)

        if result["alert_level"] in ("ORANGE","RED") or result["is_anomaly"]:
            self._fire_alert(device_id, enriched)

    def _fire_alert(self, device_id: str, snap: dict):
        ctx = {
            "device_id":               device_id,
            "health_index":            snap.get("health_index", 50),
            "rul_days":                snap.get("rul_days", 90),
            "rul_ci_low":              snap.get("rul_ci_low", 80),
            "rul_ci_high":             snap.get("rul_ci_high", 100),
            "arc_energy_sum_72h":      snap.get("arc_sum_72h", 0),
            "temperature_mean_C":      snap.get("reg_3019_temp_C", 30),
            "contact_resistance_mOhm": snap.get("reg_4004_contact_resist_mOhm", 50),
            "bounce_ms":               snap.get("reg_4003_contact_bounce_ms", 2),
            "is_anomaly":              snap.get("is_anomaly", False),
            "ac_starts_detected":      bool(snap.get("is_ac_start", 0)),
            "is_coastal":              False,
        }
        report = generate_diagnostic_report(ctx, self.retriever)
        self.store.save_report(report)
        self.store.push_alert(device_id, {
            "alert_id":    report["report_id"],
            "alert_level": snap["alert_level"],
            "health_index":snap["health_index"],
            "rul_days":    snap["rul_days"],
            "message":     report["customer"]["message"],
            "fired_at":    datetime.now().isoformat(),
        })
        # [SWAP_PUSH] FCM here: fcm.notify(token, report["customer"]["message"])
        print(f"  [ALERT] {device_id} → {snap['alert_level']} | "
              f"HI={snap['health_index']}% | RUL={snap['rul_days']}d")

    def simulate_feed(self, n: int = 20) -> Optional[str]:
        """Load real synthetic data and simulate MQTT messages."""
        csv = os.path.join(DATA_DIR, "heavy_ac_user.csv")
        if not os.path.exists(csv):
            print("  [WARN] Run Notebook 1 first")
            return None
        df = pd.read_csv(csv)
        # Pick rows at moderate degradation so alerts fire
        df = df[(df["reg_4002_relay_health_index"] < 45) &
                (df["reg_4002_relay_health_index"] > 5)].head(n)
        if df.empty:
            df = pd.read_csv(csv).tail(n)
        device_id = "INV-DEMO-001"
        print(f"  Streaming {len(df)} messages for {device_id}...")
        for _, row in df.iterrows():
            payload = row.to_dict()
            payload.pop("scenario", None)
            self.handle(device_id, payload)
            time.sleep(0.02)
        return device_id

## Section

In [ ]:
# 5. REST API

## Section

In [ ]:
def build_api(store: InMemoryStore, models: ModelStore,
              retriever: TFIDFRetriever, mqtt: MQTTHandler):
    """Build FastAPI app. Requires: pip install fastapi uvicorn pydantic"""
    try:
        from fastapi import FastAPI, HTTPException, BackgroundTasks
        from fastapi.middleware.cors import CORSMiddleware
        from pydantic import BaseModel
        import uvicorn
    except ImportError:
        print("  [INFO] FastAPI not installed — API server skipped")
        print("         pip install fastapi uvicorn pydantic")
        return None

    app = FastAPI(title="ARIA Relay Intelligence API", version="1.0.0")
    app.add_middleware(CORSMiddleware, allow_origins=["*"],
                       allow_methods=["*"], allow_headers=["*"])

    class TelemetryIn(BaseModel):
        device_id: str
        timestamp: Optional[str] = None
        reg_3002_discharge_A:         float = 0
        reg_3019_temp_C:              float = 30
        reg_3054_inv_current_A:       float = 5
        reg_3060_load_pct:            float = 50
        reg_4002_relay_health_index:  float = 100
        reg_4003_contact_bounce_ms:   float = 2
        reg_4004_contact_resist_mOhm: float = 50
        reg_4005_arc_energy_this_switch: float = 0
        is_power_cut:  int   = 0
        is_ac_start:   int   = 0
        temp_factor:   float = 1.0

    class WhatIf(BaseModel):
        device_id:           str
        extra_load_kw:       float = 0.0
        extra_ac_starts_day: int   = 0
        ambient_temp_C:      float = 35.0

    @app.get("/")
    def root():
        return {"service": "ARIA", "status": "ok", "docs": "/docs"}

    @app.get("/health")
    def api_health():
        return {"models_loaded": models.gbr is not None,
                "kb_docs": len(KB_DOCUMENTS),
                "devices": len(store.all_devices())}

    @app.post("/telemetry")
    def ingest(payload: TelemetryIn, bg: BackgroundTasks):
        data = payload.model_dump()
        data.setdefault("timestamp", datetime.now().isoformat())
        bg.add_task(mqtt.handle, payload.device_id, data)
        return {"accepted": True, "device_id": payload.device_id}

    @app.get("/devices")
    def list_devices():
        result = []
        for dev in store.all_devices():
            h = store.get_telemetry(dev, n=1)
            if h:
                result.append({"device_id": dev,
                                "health_index": h[-1].get("health_index"),
                                "alert_level":  h[-1].get("alert_level","GREEN"),
                                "rul_days":     h[-1].get("rul_days")})
        return {"devices": result, "count": len(result)}

    @app.get("/devices/{device_id}/health")
    def device_health(device_id: str):
        """Homeowner app: fuel-gauge endpoint."""
        h = store.get_telemetry(device_id, n=1)
        if not h:
            raise HTTPException(404, f"{device_id} not found")
        lat = h[-1]
        alerts = store.get_alerts(device_id)
        return {
            "device_id":    device_id,
            "health_index": lat.get("health_index"),
            "alert_level":  lat.get("alert_level","GREEN"),
            "rul_days":     lat.get("rul_days"),
            "rul_ci":       [lat.get("rul_ci_low"), lat.get("rul_ci_high")],
            "is_anomaly":   lat.get("is_anomaly", False),
            "temperature_C":lat.get("reg_3019_temp_C"),
            "open_alerts":  len([a for a in alerts
                                  if a["alert_level"] in ("ORANGE","RED")]),
        }

    @app.get("/devices/{device_id}/telemetry")
    def device_telemetry(device_id: str, n: int = 48):
        """Trend chart data — 24h at 30-min sampling."""
        h = store.get_telemetry(device_id, n=n)
        if not h:
            raise HTTPException(404, f"{device_id} not found")
        fields = ["timestamp","reg_4002_relay_health_index","rul_days",
                  "reg_3019_temp_C","reg_3060_load_pct",
                  "reg_4003_contact_bounce_ms","alert_level"]
        return {"device_id": device_id,
                "samples": [{k: r.get(k) for k in fields} for r in h]}

    @app.get("/devices/{device_id}/alerts")
    def device_alerts(device_id: str):
        return {"device_id": device_id,
                "alerts": store.get_alerts(device_id)}

    @app.get("/reports/{report_id}")
    def get_report(report_id: str):
        """Engineer app: full RAG diagnostic report."""
        r = store.get_report(report_id)
        if not r:
            raise HTTPException(404, f"Report {report_id} not found")
        return r

    @app.post("/devices/{device_id}/report")
    def on_demand_report(device_id: str, image_label: Optional[str] = None):
        """Trigger RAG report on demand (e.g. engineer opens app)."""
        h = store.get_telemetry(device_id, n=1)
        if not h:
            raise HTTPException(404, f"{device_id} not found")
        lat  = h[-1]
        hi   = lat.get("health_index", 50)
        rul  = lat.get("rul_days", 90)
        ctx  = {"device_id": device_id, "health_index": hi, "rul_days": rul,
                "rul_ci_low": max(0, rul-10), "rul_ci_high": rul+10,
                "arc_energy_sum_72h": 0, "temperature_mean_C": lat.get("reg_3019_temp_C",30),
                "contact_resistance_mOhm": lat.get("reg_4004_contact_resist_mOhm",50),
                "bounce_ms": lat.get("reg_4003_contact_bounce_ms",2),
                "is_anomaly": lat.get("is_anomaly",False),
                "ac_starts_detected": bool(lat.get("is_ac_start",0)),
                "is_coastal": False}
        img  = f"photos/{device_id}_{image_label}.jpg" if image_label else None
        rep  = generate_diagnostic_report(ctx, retriever, image_path=img)
        store.save_report(rep)
        return rep

    @app.post("/simulate/whatif")
    def what_if(req: WhatIf):
        """What-if simulator: 'If I add a 1.5-ton AC, what happens to relay life?'"""
        h   = store.get_telemetry(req.device_id, n=1)
        lat = h[-1] if h else {}
        hi  = float(lat.get("health_index", 70))

        k    = 1.2e-9
        def arrh(T): return np.exp(0.7/8.617e-5*(1/298.15 - 1/(T+273.15)))
        max_w  = 100 * k * 100 * 12 * 2.0 * 1e5
        rem_w  = max_w * (hi/100)
        base_T = float(lat.get("reg_3019_temp_C", 30))
        base_dw = 4 * k * 36 * 12 * 2.5 * arrh(base_T)
        extra_dw = (req.extra_ac_starts_day*2 * k * 225*12*3.0
                    * arrh(req.ambient_temp_C)
                    + req.extra_load_kw * k * 100*12*2.0 * arrh(req.ambient_temp_C))
        cur_rul  = rem_w / base_dw  if base_dw > 0  else 999
        new_rul  = rem_w / (base_dw+extra_dw) if (base_dw+extra_dw) > 0 else 999
        delta    = 100*(1 - new_rul/max(cur_rul,1))
        return {
            "current_rul_days":   round(cur_rul, 1),
            "projected_rul_days": round(new_rul, 1),
            "rul_reduction_pct":  round(delta, 1),
            "recommendation": (
                f"Adding {req.extra_load_kw}kW + {req.extra_ac_starts_day} AC starts/day "
                f"reduces relay life by {delta:.0f}% "
                f"({cur_rul:.0f}d → {new_rul:.0f}d)."
            )
        }

    @app.get("/fleet/overview")
    def fleet_overview():
        """Luminous dashboard: fleet health heatmap."""
        devs = store.all_devices()
        summ = {"RED":0,"ORANGE":0,"YELLOW":0,"GREEN":0}
        rows = []
        for d in devs:
            h = store.get_telemetry(d, n=1)
            if h:
                lvl = h[-1].get("alert_level","GREEN")
                summ[lvl] = summ.get(lvl,0)+1
                rows.append({"device_id":d,"alert_level":lvl,
                             "health_index":h[-1].get("health_index"),
                             "rul_days":h[-1].get("rul_days")})
        return {"fleet_size":len(devs),"summary":summ,
                "critical":summ["RED"]+summ["ORANGE"],"devices":rows}

    return app, uvicorn

## Section

In [ ]:
# 6. DEMO (no HTTP server needed)

## Section

In [ ]:
def run_demo():
    print("="*60)
    print("ARIA — Notebook 4: FastAPI + MQTT Demo")
    print("="*60)

    store     = InMemoryStore()
    models    = ModelStore()
    docs      = build_knowledge_base()
    retriever = TFIDFRetriever(docs)
    mqtt      = MQTTHandler(store, models, retriever)

    device_id = mqtt.simulate_feed(n=20)
    if not device_id:
        return

    print("\n── Simulated API Responses ─────────────────────────────")

    # GET /devices/{id}/health
    h   = store.get_telemetry(device_id, n=1)
    lat = h[-1] if h else {}
    hi  = lat.get("health_index", lat.get("reg_4002_relay_health_index", "?"))
    print(f"\n  GET /devices/{device_id}/health")
    print(f"    health_index : {hi}%")
    print(f"    alert_level  : {lat.get('alert_level','?')}")
    print(f"    rul_days     : {lat.get('rul_days','?')} d")
    print(f"    rul_ci       : [{lat.get('rul_ci_low','?')} – {lat.get('rul_ci_high','?')}]")
    print(f"    is_anomaly   : {lat.get('is_anomaly',False)}")

    # GET /devices/{id}/alerts
    alerts = store.get_alerts(device_id)
    print(f"\n  GET /devices/{device_id}/alerts → {len(alerts)} alerts")
    for a in alerts[:2]:
        print(f"    [{a['alert_level']}] {a['message'][:90]}...")

    # GET /reports/{id}
    if alerts:
        rid    = alerts[0]["alert_id"]
        report = store.get_report(rid)
        if report:
            print(f"\n  GET /reports/{rid}")
            print(f"    Failure mode : {report['alert']['failure_mode']}")
            print(f"    Severity     : {report['alert']['severity']}")
            print(f"    Engineer steps:")
            for s in report["engineer"]["actions"]:
                print(f"      • {s}")
            print(f"    KB sources   : {report['retrieval']['sources_used']}")

    # POST /simulate/whatif
    rem = float(lat.get("health_index", 30))
    k, arrh = 1.2e-9, lambda T: np.exp(0.7/8.617e-5*(1/298.15-1/(T+273.15)))
    mw  = 100*k*100*12*2.0*1e5*(rem/100)
    bdw = 4*k*36*12*2.5*arrh(30)
    edw = 8*2*k*225*12*3.0*arrh(42)
    cur_rul = mw/bdw if bdw>0 else 999
    new_rul = mw/(bdw+edw) if (bdw+edw)>0 else 999
    print(f"\n  POST /simulate/whatif  (+1.5kW AC, +8 starts/day, 42°C)")
    print(f"    Current RUL   : {cur_rul:.1f} days")
    print(f"    Projected RUL : {new_rul:.1f} days")
    print(f"    Life reduction: {100*(1-new_rul/max(cur_rul,1)):.0f}%")

    # Save state
    state = {"demo_device": device_id, "health_index": hi,
             "alerts": len(alerts), "generated_at": datetime.now().isoformat()}
    with open("aria_server_state.json","w") as f:
        json.dump(state, f, indent=2)

    print(f"\n{'='*60}")
    print("State saved → aria_server_state.json")
    print("\nTo start the real API server:")
    print("  pip install fastapi uvicorn pydantic")
    print("  python notebook4_api_server.py --server")
    print("  Open: http://localhost:8000/docs")
    print("Next step → Notebook 5: React web app (mobile-first UI)")
    print("="*60)


if __name__ == "__main__":
    if "--server" in sys.argv:
        store     = InMemoryStore()
        models    = ModelStore()
        docs      = build_knowledge_base()
        retriever = TFIDFRetriever(docs)
        mqtt      = MQTTHandler(store, models, retriever)
        result    = build_api(store, models, retriever, mqtt)
        if result:
            app, uvicorn = result
            uvicorn.run(app, host="0.0.0.0", port=8000)
    else:
        run_demo()

## Run Demo (no HTTP server needed)

In [ ]:
run_demo()

## Start Live API Server

Uncomment and run to start the FastAPI server.
Then open **http://localhost:8000/docs** for the interactive Swagger UI.

In [ ]:
# import uvicorn
# store=InMemoryStore(); models=ModelStore()
# docs=build_knowledge_base(); retriever=TFIDFRetriever(docs)
# mqtt=MQTTHandler(store,models,retriever)
# result=build_api(store,models,retriever,mqtt)
# if result:
#     app_obj,uv=result
#     uv.run(app_obj, host='0.0.0.0', port=8000)